A continuación te dejo un modelo de segmentación de imágenes usando una arquitectura U-Net para segmentar imágenes satelitales. Puedes usar datos satelitales (GeoTIFF) y su correspondiente máscara de segmentación.

In [ ]:
# Importación de librerías
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import rasterio

In [ ]:
# Cargar las imágenes de GeoTIFF
def load_geotiff(image_path):
    with rasterio.open(image_path) as src:
        img_array = src.read([1, 2, 3])  # Leer las bandas RGB
        img_array = np.moveaxis(img_array, 0, -1)  # Cambiar el orden para que sea (height, width, channels)
    return img_array

In [ ]:
# Definir el modelo U-Net
def unet_model(input_size=(128, 128, 3)):
    inputs = layers.Input(input_size)

    # Encoder
    conv1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    conv1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(conv1)
    pool1 = layers.MaxPooling2D((2, 2))(conv1)

    conv2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(pool1)
    conv2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(conv2)
    pool2 = layers.MaxPooling2D((2, 2))(conv2)

    # Bottleneck
    conv3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(pool2)
    conv3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(conv3)

    # Decoder
    up1 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv3)
    concat1 = layers.concatenate([up1, conv2])
    conv4 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(concat1)
    conv4 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(conv4)

    up2 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv4)
    concat2 = layers.concatenate([up2, conv1])
    conv5 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(concat2)
    conv5 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(conv5)

    # Output layer
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(conv5)

    model = models.Model(inputs, outputs)
    return model

In [ ]:
# Compilar el modelo
model = unet_model(input_size=(128, 128, 3))

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Cargar datos de entrenamiento
train_images = np.array([load_geotiff('path/to/image.tif')])
train_masks = np.array([load_geotiff('path/to/mask.tif')])

In [ ]:
# Entrenar el modelo
model.fit(train_images, train_masks, epochs=5, batch_size=1)

Instrucciones de Ejecución:

* Subir datos: Si trabajas en Google Colab, puedes cargar tus datos desde Google Drive o usar comandos como !wget para descargar los datos.

* Rutas de imágenes: Asegúrate de ajustar las rutas ('path/to/your/image.tif') con las ubicaciones correctas de tus imágenes y máscaras.

* Entrenamiento: Estos códigos están listos para entrenar, pero necesitarás tener un conjunto adecuado de imágenes de entrada y etiquetas de salida.